# 01 — Data Ingest

Pulls 5 seasons of NFL data from nflverse (via `nflreadpy`) and caches to `data/raw/` as Parquet.

Run this **once at season start**, then re-run weekly to refresh the current season.

In [ ]:
import nflreadpy as nfl
import polars as pl
from pathlib import Path

SEASONS = [2021, 2022, 2023, 2024, 2025]   # 5 most recent completed seasons
DATA_DIR = Path('../data/raw')
DATA_DIR.mkdir(parents=True, exist_ok=True)
print('Pulling seasons:', SEASONS)

## Schedules (results + closing lines)

In [ ]:
schedules = nfl.load_schedules(seasons=SEASONS)
schedules.write_parquet(DATA_DIR / 'schedules.parquet')
print(f'Schedules rows: {schedules.shape[0]:,} | cols: {schedules.shape[1]}')
schedules.select(['game_id','season','week','home_team','away_team','home_score','away_score','spread_line','total_line']).head(10)

## Play-by-play (EPA, success, all snaps)

In [ ]:
# ~50k plays/season, this can take 1-2 min on first run (cached after)
pbp = nfl.load_pbp(seasons=SEASONS)
pbp.write_parquet(DATA_DIR / 'pbp.parquet')
print(f'PBP rows: {pbp.shape[0]:,} | cols: {pbp.shape[1]}')

## Team metadata

In [ ]:
teams = nfl.load_teams()
teams.write_parquet(DATA_DIR / 'teams.parquet')
print(f'Teams: {teams.shape[0]}')
teams.select(['team_abbr','team_name','team_conf','team_division']).head()

## Done

Files written to `data/raw/`. Next: open `02_features.ipynb`.